# Session 11: Bootcamp: Segmenting a Wholesaler's Customers

## Overview

A wholesale food distributor in Portugal supplies 440 business customers: restaurants, hotels and cafés on one
side, shops and supermarkets on the other. It wants to **segment** them, so that delivery schedules, sales visits
and offers fit each kind of customer, instead of treating everyone alike. There is no "right answer" in the data
this time; the segments have to be found, judged and named.

This bootcamp applies Session 10 to a real problem and adds what real segmentation needs: transforming skewed
spending data, choosing the number of segments when the guides disagree, checking that the segments are **stable**
by comparing two different algorithms (k-means and **hierarchical clustering**), describing each segment in
business terms, and finding the unusual customers that fit no segment.

By the end, you will be able to:

- Prepare skewed monetary data for clustering with a log transform and scaling.
- Choose the number of clusters with the silhouette score *and* business judgement.
- Run agglomerative (hierarchical) clustering with Ward linkage and read a dendrogram.
- Check the stability of a segmentation by comparing two algorithms with the adjusted Rand index.
- Profile and name segments, and flag atypical customers by their distance to the centroid.

**Prerequisites:** Session 10 (k-means, scaling, silhouette, ARI).

**Dataset:** [UCI Wholesale Customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers). 440 clients
of a wholesale distributor in Portugal (Cardoso, 2014): annual spending (in monetary units) on six product
categories (fresh, milk, grocery, frozen, detergents and paper, delicatessen), the **channel** (1 = hotel,
restaurant or café, "Horeca"; 2 = retail) and the region (1 = Lisbon, 2 = Oporto, 3 = other). It is downloaded
automatically by the code.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering **← you are here** |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** What kinds of customers does the wholesaler have, judging by what they buy?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| ucimlrepo | Downloads the dataset from UCI |
| scikit-learn | Scaling, k-means, agglomerative clustering, metrics |
| scipy | The dendrogram |
| pandas | Tables |
| numpy | Log transform, distances |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from importlib.metadata import version

for package in ["ucimlrepo", "scikit-learn", "scipy", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the customers

We cluster on the **six spending columns only**. Channel and region are set aside: they are facts about the
customer we can use afterwards to interpret the segments, but the point is to find groups from **buying
behaviour**.

In [ ]:
from ucimlrepo import fetch_ucirepo

customers = fetch_ucirepo(id=292).data.original
SPEND = ["Fresh", "Milk", "Grocery", "Frozen", "Detergents_Paper", "Delicassen"]
spend = customers[SPEND]
print(f"{len(customers)} customers")
print("channel:", customers["Channel"].map({1: "Horeca", 2: "retail"}).value_counts().to_dict())
pd.set_option("display.width", 120)
print(spend.describe().round(0).T[["mean", "50%", "min", "max"]])

**Read the output:** 298 Horeca and 142 retail customers. Every category is heavily **skewed**: the mean is well
above the median (fresh: 12,000 against 8,504), and the maximum is many times the median (delicatessen: 47,943
against 966). A few very large customers would dominate any distance-based method.

## Step 3: Log-transform, then scale

Spending varies multiplicatively: one restaurant buys twice as much as another, a supermarket ten times as much.
A **log transform** turns ratios into differences, so "twice as much" is the same step whether it is 1,000 to 2,000
or 10,000 to 20,000, and it pulls in the extreme values. Then we standardise, as in Session 10.

In [ ]:
log_spend = np.log(spend)
print("skewness before:", spend.skew().round(1).to_dict())
print("skewness after: ", log_spend.skew().round(1).to_dict())
Z = StandardScaler().fit_transform(log_spend)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
spend["Grocery"].plot.hist(bins=40, ax=axes[0], title="Grocery spending")
log_spend["Grocery"].plot.hist(bins=40, ax=axes[1], title="log(Grocery spending)")
plt.show()

**Read the output:** skewness drops from between 2.6 and 11.2 to between about -1.6 and -0.2. The log histogram is
much closer to a symmetric hump. (A skewness of 0 is symmetric. The log slightly over-corrects fresh and
delicatessen: a few customers spend almost nothing on them, which become long *left* tails.)

## Step 4: How many segments?

In [ ]:
rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=20, random_state=0).fit(Z)
    rows.append({"k": k, "inertia": model.inertia_, "silhouette": silhouette_score(Z, model.labels_),
                 "sizes": np.sort(np.bincount(model.labels_))[::-1].tolist()})
print(pd.DataFrame(rows).set_index("k").round(3).to_string())

**Read the output:** this is what real data usually looks like. The silhouette score is highest at **k = 2**
(0.290), then falls to 0.258 at k = 3 and about 0.19 from k = 4 on. There is no sharp elbow in the inertia. The
data has two well-separated groups, and less clear structure beyond that.

Two segments may be too coarse for the business: the sales team wants to tell apart more kinds of customer. But
segments that the data does not really support will not be stable, and will not be useful. Steps 5-7 compare
k = 2, 3 and 4 by asking a practical question: **would a different algorithm find the same segments?**

## Step 5: Hierarchical clustering

**Agglomerative (hierarchical) clustering** works bottom-up: start with every customer in its own cluster, and
repeatedly **merge the two closest clusters** until one remains. The full sequence of merges is a tree, the
**dendrogram**; cutting it at some height gives any number of clusters.

"Closest" needs a definition, the **linkage**. **Ward** linkage merges the pair whose merger increases the total
within-cluster sum of squares the least: the same quantity k-means minimises, but found greedily, step by step,
with no random start. It is a natural partner for k-means.

In [ ]:
merges = linkage(Z, method="ward")
fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(merges, truncate_mode="lastp", p=30, ax=ax, color_threshold=(merges[-3, 2] + merges[-2, 2]) / 2)
ax.set_xlabel("customers (numbers in brackets: how many are in that branch)")
ax.set_ylabel("merge distance (Ward)")
ax.set_title("The last 30 merges of the hierarchy")
plt.show()
print("heights of the last five merges:", merges[-5:, 2].round(1))

**Read the output:** read the dendrogram from the top. The **last merge**, joining the two main branches, happens
at a height of 37.8, far above the next one (24.5): the two top-level groups are far apart, confirming the
silhouette's preference for k = 2. The next splits down happen at lower, closer heights. Cutting just below the second-last
merge gives three clusters (the colours).

## Step 6: Do k-means and Ward agree?

If two algorithms that work very differently produce the same segments, those segments reflect real structure.
If they disagree, the segmentation depends on the method more than on the data. Compare them with the ARI from
Session 10.

In [ ]:
for k in [2, 3, 4]:
    km = KMeans(n_clusters=k, n_init=20, random_state=0).fit_predict(Z)
    ward = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Z)
    print(f"k = {k}: agreement between k-means and Ward, ARI = {adjusted_rand_score(km, ward):.2f}")

**Read the output:** at k = 2 and k = 3 the two methods broadly agree (ARI 0.63 and 0.69). At **k = 4** the
agreement collapses (0.32): the four-way split depends on which algorithm you use, so it is not something
the business should build delivery schedules around.

We choose **k = 3**: the data supports it, two methods agree on it, and it gives the sales team one more segment
than the obvious two.

## Step 7: Profile and name the segments

Describe each segment by its **median** spending per category (medians, because spending is skewed) in the
original money units, and compare with the customers' channel.

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=20, random_state=0).fit(Z)
segment = kmeans.labels_
profile = spend.groupby(segment).median().round(0)
profile["customers"] = np.bincount(segment)
profile["retail share"] = customers.groupby(segment)["Channel"].apply(lambda c: (c == 2).mean()).round(2)
print(profile.to_string())

profile[SPEND].T.plot.bar(figsize=(11, 4), logy=True)
plt.ylabel("median annual spending (log scale)")
plt.title("What each segment buys")
plt.show()

**Read the output:** three clearly different kinds of customer:

| Segment | Customers | Spending pattern (medians) | Retail share | A name |
| --- | --- | --- | --- | --- |
| 2 | 211 | Fresh 9,612 and frozen 2,121; little milk, grocery or detergents | 1% | **Food service**: restaurants, cafés, hotel kitchens |
| 0 | 81 | Grocery 10,487, milk and detergents/paper; very little fresh (1,454) or frozen (388) | 59% | **Packaged-goods buyers**: shops and convenience stores |
| 1 | 148 | **High in every category**, including fresh (12,166) and delicatessen (1,988) | 61% | **Large all-round buyers**: supermarkets, larger hotels |

These are segments a salesperson recognises, which is the real test. The channel was never used to build them,
yet the food-service segment is 99% Horeca. The other two segments mix both channels: a large hotel can buy like a
supermarket. Buying behaviour says something the channel label alone does not.

(The segment numbers are arbitrary: a different `random_state` could number the same segments differently.)

## Step 8: A picture of the segments

In [ ]:
pca = PCA(n_components=2).fit(Z)
picture = pca.transform(Z)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(*picture.T, c=segment, s=14, cmap="viridis")
axes[0].set_title("k-means segments (k = 3)")
axes[1].scatter(*picture.T, c=customers["Channel"], s=14, cmap="coolwarm")
axes[1].set_title("Channel: blue = Horeca, red = retail")
for ax in axes:
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
plt.show()
print(f"the picture keeps {pca.explained_variance_ratio_.sum():.0%} of the variation")

**Read the output:** the picture keeps 71% of the variation. The segments occupy different regions of it, and the channel colours show the same
left-right division. There are no gaps between the segments: customers form a **continuum** from "pure
restaurant" to "pure shop", and k-means draws lines across it. Segments are a useful simplification, not natural
species. Customers near the borders could reasonably belong to either neighbour.

## Step 9: Customers that fit no segment

k-means assigns *every* customer to a segment, even one that resembles none. The distance from a customer to its
segment's centroid flags the unusual ones: worth a look by a human, and sometimes a sign of data errors.

In [ ]:
distance = np.linalg.norm(Z - kmeans.cluster_centers_[segment], axis=1)
unusual = customers.assign(segment=segment, distance=distance.round(2)).nlargest(5, "distance")
print(unusual[SPEND + ["Channel", "segment", "distance"]].to_string())
print(f"\ntypical distance (median): {np.median(distance):.2f}")

**Read the output:** the five most unusual customers are 3-4 times further from their centroid than the median
customer (1.56). Their spending has a strange shape: customer 75 spends 20,398 on fresh food but just **3** on
grocery and 3 on detergents; customer 154 spends almost nothing on anything; customer 338 spends 3 on fresh food
but 15,601 on frozen; customer 142 spends 37,036 on fresh food and 3 on delicatessen. Some could be specialist
businesses (a fishmonger, an ice-cream parlour); spending of 3 a year in a category might be a data-entry error. Either way, these customers should not drive decisions about their
segment, and are worth checking with the sales team.

## Bootcamp summary

| Step | Decision | Why |
| --- | --- | --- |
| Inputs | Six spending columns; channel and region kept for interpretation | Segment on behaviour |
| Transform | log, then standardise | Spending is skewed and multiplicative |
| Number of segments | 3 | Silhouette prefers 2; 3 is supported and stable; 4 is not |
| Stability | k-means vs Ward, ARI 0.63-0.69 at k = 2-3, 0.32 at k = 4 | Segments that depend on the method are not trustworthy |
| Result | Food service, packaged-goods buyers, large all-round buyers | Recognisable; food service is 99% Horeca |
| Follow-up | Five atypical customers to check | Distance to centroid |

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Clustering raw monetary values | A handful of huge customers form their own clusters | Log-transform, then scale |
| Taking the k with the best silhouette without thinking | Segments too coarse (or too fine) for their use | Combine the score with business needs and stability |
| Trusting one algorithm's clusters | Segments may be artefacts of the method | Compare two methods (ARI); try different seeds |
| Profiling with means on skewed data | One big customer drags the mean | Use medians |
| Treating segments as natural kinds | Hard boundaries on a continuum | Report border cases; use segments as a simplification |
| Ignoring atypical customers | Errors or special cases distort a segment | Flag large distances to the centroid; review them |

## Exercises

- [ ] Cluster on spending **shares** (each category divided by the customer's total) instead of amounts. Which
  segments appear? When would a business prefer shares to amounts?
- [ ] Repeat Step 6 with `linkage="average"` and `linkage="complete"` instead of Ward. How well do they agree with
  k-means? Look at the cluster sizes.
- [ ] Drop the five unusual customers from Step 9 and re-run Steps 4 and 7. Do the segments change?
- [ ] Cross-tabulate the three segments with `Region`. Does location matter for buying behaviour?

This ends Part 3. Next, Session 12 finds structure in text: topic modelling with LDA, which treats each document
as a mixture of topics.